# Training-data explorer for J-lens causal experiments

This notebook is a read-only map of what data was used to fit each
intervention and what data is only used for causal evaluation. It does
not load a model or allocate GPU memory.

The key distinction is:

- **J-lens:** a learned Jacobian/readout map. The released 27B lens
  metadata is available, but its original prompt text is not in this
  checkout. Our smaller local J-lens prompt files are inspectable.
- **Logit lens:** no fitted data; it uses the model's unembedding rows.
- **Tuned lens:** a learned affine translator. Our artifacts record a
  Wikitext-2 training source and fit settings.
- **Random matched:** no fitted data; it matches perturbation norm.

The causal fixtures are shown separately because they are evaluation
prompts, not lens-training data.

In [2]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("/home/alex/Code/jlens/jacobian-lens")
print("Repository:", ROOT)

Repository: /home/alex/Code/jlens/jacobian-lens


## 1. Intervention data inventory

In [3]:
inventory = [
    {
        "intervention": "J-lens",
        "fit_data": "Prompt corpus; released 27B prompt text unavailable locally",
        "local_artifact": "data/lenses/*jacobian* plus data/lens-prompts/ for smaller local fits",
        "trained": True,
    },
    {
        "intervention": "Logit lens",
        "fit_data": "None",
        "local_artifact": "Model unembedding matrix",
        "trained": False,
    },
    {
        "intervention": "Tuned lens",
        "fit_data": "Salesforce/wikitext, wikitext-2-raw-v1, train",
        "local_artifact": "data/lenses/*tuned-wikitext*/fit_manifest.json",
        "trained": True,
    },
    {
        "intervention": "Random matched",
        "fit_data": "None",
        "local_artifact": "Runtime-generated norm-matched random vector",
        "trained": False,
    },
]
display(inventory)

[{'intervention': 'J-lens',
  'fit_data': 'Prompt corpus; released 27B prompt text unavailable locally',
  'local_artifact': 'data/lenses/*jacobian* plus data/lens-prompts/ for smaller local fits',
  'trained': True},
 {'intervention': 'Logit lens',
  'fit_data': 'None',
  'local_artifact': 'Model unembedding matrix',
  'trained': False},
 {'intervention': 'Tuned lens',
  'fit_data': 'Salesforce/wikitext, wikitext-2-raw-v1, train',
  'local_artifact': 'data/lenses/*tuned-wikitext*/fit_manifest.json',
  'trained': True},
 {'intervention': 'Random matched',
  'fit_data': 'None',
  'local_artifact': 'Runtime-generated norm-matched random vector',
  'trained': False}]

## 2. Local J-lens prompt corpora

In [4]:
prompt_dir = ROOT / "data" / "lens-prompts"
for path in sorted(prompt_dir.glob("*.json")):
    payload = json.loads(path.read_text())
    items = payload.get("items", [])
    counts = Counter(item.get("category", "unknown") for item in items)
    print(f"{path.relative_to(ROOT)}: {len(items)} prompts; categories={dict(counts)}")
    for item in items[:3]:
        print(" -", item.get("name"), ":", item.get("prompt"))
    print()

data/lens-prompts/eval-mix-150.json: 150 prompts; categories={'typo': 25, 'multihop': 25, 'order_ops': 25, 'multilingual': 25, 'association': 25, 'poetry': 25}
 - typo-business : After college she moved to the city and started her own buisness
 - nhop-insect-element : Count the items in a dozen. The element at that position on the periodic table is magnesium. Count the legs on an insect. The element at that position on the periodic table is 
 - typo-together : The family finally sat down to eat togther

data/lens-prompts/fit-mix-120.json: 120 prompts; categories={'typo': 20, 'order_ops': 20, 'association': 20, 'multilingual': 20, 'multihop': 20, 'poetry': 20}
 - typo-library : Please carefully consider the following task and infer the relevant answer from the context before completing it. She returned the overdue books to the local libary
 - square-sub : Please carefully consider the following task and infer the relevant answer from the context before completing it. (7 - 4)^2 = 
 - add

These are our locally assembled prompt mixes for smaller model-matched
fits. They include typo correction, associations, transformations,
factual/reasoning-style prompts, and other categories. They are not
evidence that the released Anthropic 27B J-lens used the same corpus.

## 3. Tuned-lens fit manifests

In [5]:
manifests = []
for path in sorted((ROOT / "data" / "lenses").glob("*/fit_manifest.json")):
    manifest = json.loads(path.read_text())
    manifests.append({
        "artifact": str(path.parent.relative_to(ROOT)),
        "model": manifest.get("model"),
        "dataset": manifest.get("dataset"),
        "split": manifest.get("split"),
        "chunks": manifest.get("max_chunks"),
        "length": manifest.get("max_length"),
        "steps": manifest.get("steps"),
        "objective": manifest.get("objective"),
        "precision": manifest.get("quantization", manifest.get("dtype")),
    })
display(manifests)

[{'artifact': 'data/lenses/qwen3-0.6b-modal-fast-smoke-v2-verified',
  'model': 'Qwen/Qwen3-0.6B',
  'dataset': 'Salesforce/wikitext/wikitext-2-raw-v1',
  'split': 'train',
  'chunks': 1,
  'length': 32,
  'steps': 1,
  'objective': 'per-layer KL to frozen final logits',
  'precision': 'nf4'},
 {'artifact': 'data/lenses/qwen3-0.6b-nvidia-vanilla-4bit-smoke',
  'model': 'Qwen/Qwen3-0.6B',
  'dataset': 'Salesforce/wikitext/wikitext-2-raw-v1',
  'split': 'train',
  'chunks': 1,
  'length': 32,
  'steps': 1,
  'objective': 'per-layer KL to frozen final logits',
  'precision': 'nf4'},
 {'artifact': 'data/lenses/qwen3-0.6b-nvidia-vanilla-8bit-smoke',
  'model': 'Qwen/Qwen3-0.6B',
  'dataset': 'Salesforce/wikitext/wikitext-2-raw-v1',
  'split': 'train',
  'chunks': 1,
  'length': 32,
  'steps': 1,
  'objective': 'per-layer KL to frozen final logits',
  'precision': 'int8'},
 {'artifact': 'data/lenses/qwen3-0.6b-nvidia-vanilla-image-smoke',
  'model': 'Qwen/Qwen3-0.6B',
  'dataset': 'Salesforc

For the 27B tuned fit specifically, the manifest says: 512 Wikitext
chunks, maximum length 128, 100 steps, BF16 compute over NF4 weights,
and per-layer KL divergence to the frozen final logits. Wikitext is
raw text; it is not the same thing as our causal reasoning fixtures.

In [6]:
# Optional: fetch/display raw Wikitext examples if `datasets` is installed.
# This cell may access the Hugging Face cache or network; it is not needed
# to inspect the local metadata above.
try:
    from datasets import load_dataset
    wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    rows = [row["text"] for row in wiki if row["text"].strip()][:5]
    for i, row in enumerate(rows, 1):
        print(f"Wikitext example {i}: {row[:500]!r}")
except Exception as exc:
    print("Wikitext loading skipped:", type(exc).__name__, exc)

Wikitext loading skipped: ModuleNotFoundError No module named 'datasets'


## 4. Causal evaluation fixtures (not training data)

In [ ]:
fixtures = {
    "verbal report": ROOT / "data/experiments/verbal-report.json",
    "two-hop reasoning": ROOT / "data/experiments/probe-swap.json",
    "flexible generalization": ROOT / "data/experiments/flexible-generalization.json",
    "conjunction candidates": ROOT / "data/benchmarks/proofwriter-strong-conjunctions.jsonl",
}
for name, path in fixtures.items():
    if path.suffix == ".jsonl":
        rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        print(f"{name}: {len(rows)} rows from {path.relative_to(ROOT)}")
        for row in rows[:2]:
            print(" -", row.get("prompt", row))
    else:
        payload = json.loads(path.read_text())
        rows = payload.get("items", payload.get("categories", payload))
        print(f"{name}: {len(rows) if hasattr(rows, '__len__') else 'structured'} from {path.relative_to(ROOT)}")
        if isinstance(rows, list):
            for row in rows[:2]:
                print(" -", row.get("prompt", row))
    print()

These fixtures define what the model is asked and how success is
scored. They should not be described as the training corpus for any
lens unless a fit command explicitly points to them.

## Questions to investigate next

1. Do the smaller J-lens prompt mixes contain enough category and
   linguistic diversity for the intended claim?
2. Should the tuned lens use raw Wikitext, task-like prompts, or both?
3. Should we reserve a held-out corpus for fit-quality checks before
   comparing causal effectiveness?
4. Can we obtain the original released J-lens fitting prompts or only
   reproduce their reported count and broad corpus description?